# Exercises — Trading signals

[DataCamp exercise](https://campus.datacamp.com/courses/financial-trading-in-python/trading-strategies?ex=1) · see `Notes.md` in this folder for the summary.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project's data folder regardless of where this notebook runs from
DATA = next(p / "course materials" / "data"
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course materials" / "data").is_dir())

def load(name):
    """Load an OHLCV CSV with a parsed DatetimeIndex."""
    return pd.read_csv(DATA / name, index_col="Date", parse_dates=True)

def price(name, col, year=None):
    """Single-asset price DataFrame (column = `col`) for use with bt."""
    df = load(name)
    if year:
        df = df[df.index.year == year]
    return df["Close"].rename(col).to_frame()


### Long when price is above its SMA (bt `SelectWhere`)

In [ ]:
import bt
import talib

data = price("AMZN-stock-data.csv", "AMZN", year=2020)
sma = talib.SMA(data["AMZN"], timeperiod=20)

# Boolean signal DataFrame: True where price > SMA
signal = pd.DataFrame(data["AMZN"].values > sma.values,
                      index=data.index, columns=["AMZN"])

bt_strategy = bt.Strategy("AboveSMA", [
    bt.algos.SelectWhere(signal),
    bt.algos.WeighEqually(),
    bt.algos.Rebalance(),
])
bt_result = bt.run(bt.Backtest(bt_strategy, data))
bt_result.plot(title="Price > SMA signal")
plt.show()